In [1]:
# ============================================================================
# SECTION 1: IMPORTS AND SETUP
# ============================================================================
print("="*80)
print("BERT-ONLY CLASSIFIER - SEMI-SUPERVISED SETUP (10% LABELED)")
print("="*80)

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from torch.optim import AdamW  # ✅ FIXED: Import from torch.optim, not transformers
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import warnings
import time  # ✅ ADDED: For timing
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n✓ Device: {device}")
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✓ Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

BERT-ONLY CLASSIFIER - SEMI-SUPERVISED SETUP (10% LABELED)

✓ Device: cuda
✓ GPU: NVIDIA GeForce RTX 3050 Laptop GPU
✓ Memory: 4.29 GB


In [2]:
# ============================================================================
# SECTION 2: CONFIGURATION
# ============================================================================
print("\n[2/9] Configuration...")

class Config:
    # Model settings
    BERT_MODEL = 'google/muril-base-cased'
    BERT_DIM = 768
    HIDDEN_DIM = 256
    DROPOUT = 0.3
    
    # Training settings
    MAX_LENGTH = 256
    BATCH_SIZE = 8  # ✅ FIXED: Reduced from 16 for 4GB GPU
    LEARNING_RATE = 1e-5  # Very low LR as recommended
    EPOCHS = 20
    WARMUP_RATIO = 0.1
    WEIGHT_DECAY = 0.01
    LABEL_SMOOTHING = 0.1
    
    # Early stopping
    EARLY_STOPPING_PATIENCE = 5
    
    # Semi-supervised setup
    LABELED_RATIO = 0.10  # Only 10% labeled for training
    VAL_RATIO = 0.05      # 5% for validation
    TEST_RATIO = 0.85     # Remaining 85% for testing (simulating unlabeled)
    SEED = 42
    
    # Fine-tuning strategy (freeze layers 1-8, fine-tune 9-12)
    FREEZE_BERT_LAYERS = 8

torch.manual_seed(Config.SEED)
np.random.seed(Config.SEED)

print(f"✓ Model: {Config.BERT_MODEL}")
print(f"✓ Labeled Data: {Config.LABELED_RATIO*100:.0f}%")
print(f"✓ Validation Data: {Config.VAL_RATIO*100:.0f}%")
print(f"✓ Test Data: {Config.TEST_RATIO*100:.0f}%")
print(f"✓ Freeze Layers: {Config.FREEZE_BERT_LAYERS}/12 (train layers 9-12)")
print(f"✓ Learning Rate: {Config.LEARNING_RATE}")
print(f"✓ Batch Size: {Config.BATCH_SIZE}")
print(f"✓ Epochs: {Config.EPOCHS}")



[2/9] Configuration...
✓ Model: google/muril-base-cased
✓ Labeled Data: 10%
✓ Validation Data: 5%
✓ Test Data: 85%
✓ Freeze Layers: 8/12 (train layers 9-12)
✓ Learning Rate: 1e-05
✓ Batch Size: 8
✓ Epochs: 20


In [3]:
# ============================================================================
# SECTION 3: BERT-ONLY MODEL
# ============================================================================
print("\n[3/9] Defining BERT-Only model...")

class BERTClassifier(nn.Module):
    """BERT-based classifier with regularization"""
    
    def __init__(self, bert_model, num_labels, hidden_dim=256, dropout=0.5):
        super().__init__()
        self.bert = bert_model
        
        # Freeze early BERT layers to prevent overfitting
        if Config.FREEZE_BERT_LAYERS > 0:
            # Freeze embeddings
            for param in self.bert.embeddings.parameters():
                param.requires_grad = False
            
            # Freeze first N encoder layers
            for layer in self.bert.encoder.layer[:Config.FREEZE_BERT_LAYERS]:
                for param in layer.parameters():
                    param.requires_grad = False
            print(f"✓ Froze embeddings + first {Config.FREEZE_BERT_LAYERS}/12 BERT layers")
        
        self.dropout = nn.Dropout(dropout)
        
        # Enhanced classification head with layer normalization
        self.classifier = nn.Sequential(
            nn.Linear(Config.BERT_DIM, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_labels)
        )
    
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )
        
        cls_output = outputs.last_hidden_state[:, 0, :]
        cls_output = self.dropout(cls_output)
        logits = self.classifier(cls_output)
        
        return logits

print("✓ BERTClassifier defined")



[3/9] Defining BERT-Only model...
✓ BERTClassifier defined


In [4]:
# ============================================================================
# SECTION 4: DATASET
# ============================================================================
print("\n[4/9] Defining dataset...")

class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(label, dtype=torch.long)
        }

print("✓ NewsDataset defined")



[4/9] Defining dataset...
✓ NewsDataset defined


In [5]:
# ============================================================================
# SECTION 5: DATA LOADING
# ============================================================================
print("\n[5/9] Loading data...")

# Load data
df = pd.read_csv('preprocessed_news02.csv')
print(f"✓ Loaded: {df.shape[0]} samples")

# Encode labels
le = LabelEncoder()
if 'clean_categories' in df.columns:
    labels = le.fit_transform(df['clean_categories'].astype(str))
else:
    labels = le.fit_transform(df['category'].astype(str))
    
num_labels = len(le.classes_)

print(f"✓ Categories: {num_labels}")
print(f"✓ Classes: {list(le.classes_)}")



[5/9] Loading data...
✓ Loaded: 122817 samples
✓ Categories: 10
✓ Classes: ['art', 'crime', 'economy', 'education', 'entertainment', 'global', 'health', 'politics', 'science and technology', 'sports']


In [6]:
# ============================================================================
# SECTION 6: SEMI-SUPERVISED SPLIT (10% LABELED)
# ============================================================================
print("\n[6/9] Creating semi-supervised splits...")

# Get all indices
all_idxs = np.arange(len(df))

# First split: 10% labeled, 90% unlabeled (which we'll use as test)
labeled_idx, unlabeled_idx = train_test_split(
    all_idxs,
    test_size=Config.TEST_RATIO,
    random_state=Config.SEED,
    stratify=labels
)

# From the 10% labeled, split into train and val
# Val will be 5% of total (so 1/3 of the labeled data)
train_idx, val_idx = train_test_split(
    labeled_idx,
    test_size=Config.VAL_RATIO / (Config.LABELED_RATIO + Config.VAL_RATIO),
    random_state=Config.SEED,
    stratify=labels[labeled_idx]
)

# Use the "unlabeled" data as test set
test_idx = unlabeled_idx

print(f"\n{'='*70}")
print(f"SEMI-SUPERVISED DATA SPLIT")
print(f"{'='*70}")
print(f"🟢 Train (labeled):   {len(train_idx):6d} ({len(train_idx)/len(df)*100:5.2f}%)")
print(f"🟡 Val (labeled):     {len(val_idx):6d} ({len(val_idx)/len(df)*100:5.2f}%)")
print(f"🔵 Test (unlabeled):  {len(test_idx):6d} ({len(test_idx)/len(df)*100:5.2f}%)")
print(f"{'='*70}")
print(f"Total Labeled:        {len(train_idx) + len(val_idx):6d} ({(len(train_idx) + len(val_idx))/len(df)*100:5.2f}%)")
print(f"Total Dataset:        {len(df):6d}")
print(f"{'='*70}\n")

# Create masks for future GCN use
train_mask = np.zeros(len(df), dtype=bool)
val_mask = np.zeros(len(df), dtype=bool)
test_mask = np.zeros(len(df), dtype=bool)

train_mask[train_idx] = True
val_mask[val_idx] = True
test_mask[test_idx] = True

# Save masks for GCN
np.save('train_mask.npy', train_mask)
np.save('val_mask.npy', val_mask)
np.save('test_mask.npy', test_mask)
print("✓ Saved train/val/test masks for GCN")



[6/9] Creating semi-supervised splits...

SEMI-SUPERVISED DATA SPLIT
🟢 Train (labeled):    12281 (10.00%)
🟡 Val (labeled):       6141 ( 5.00%)
🔵 Test (unlabeled):  104395 (85.00%)
Total Labeled:         18422 (15.00%)
Total Dataset:        122817

✓ Saved train/val/test masks for GCN


In [7]:
# ============================================================================
# SECTION 7: MODEL INITIALIZATION
# ============================================================================
print("\n[7/9] Initializing model...")

tokenizer = AutoTokenizer.from_pretrained(Config.BERT_MODEL)
bert_model = AutoModel.from_pretrained(Config.BERT_MODEL)

# Create datasets (only for labeled data)
train_dataset = NewsDataset(
    df['content'].iloc[train_idx].values,
    labels[train_idx],
    tokenizer,
    Config.MAX_LENGTH
)

val_dataset = NewsDataset(
    df['content'].iloc[val_idx].values,
    labels[val_idx],
    tokenizer,
    Config.MAX_LENGTH
)

test_dataset = NewsDataset(
    df['content'].iloc[test_idx].values,
    labels[test_idx],
    tokenizer,
    Config.MAX_LENGTH
)

# ✅ FIXED: Added num_workers=0 for Windows compatibility
train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=Config.BATCH_SIZE, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=Config.BATCH_SIZE, num_workers=0)

print(f"✓ Train batches: {len(train_loader)}")
print(f"✓ Val batches: {len(val_loader)}")
print(f"✓ Test batches: {len(test_loader)}")

# ✅ ADDED: Time estimate
print(f"✓ Estimated time per epoch: ~{len(train_loader) * 2 / 60:.1f} minutes")

# Initialize model
model = BERTClassifier(
    bert_model=bert_model,
    num_labels=num_labels,
    hidden_dim=Config.HIDDEN_DIM,
    dropout=Config.DROPOUT
).to(device)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"✓ Trainable parameters: {trainable_params:,} / {total_params:,} ({100*trainable_params/total_params:.1f}%)")

# Optimizer and scheduler
optimizer = AdamW(model.parameters(), lr=Config.LEARNING_RATE, weight_decay=Config.WEIGHT_DECAY)

num_training_steps = len(train_loader) * Config.EPOCHS
num_warmup_steps = int(num_training_steps * Config.WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)

# Label Smoothing Loss
criterion = nn.CrossEntropyLoss(label_smoothing=Config.LABEL_SMOOTHING)

print("✓ Setup complete")


[7/9] Initializing model...
✓ Train batches: 1536
✓ Val batches: 768
✓ Test batches: 13050
✓ Estimated time per epoch: ~51.2 minutes
✓ Froze embeddings + first 8/12 BERT layers
✓ Trainable parameters: 29,173,898 / 237,788,042 (12.3%)
✓ Setup complete


In [8]:
# ============================================================================
# SECTION 8: TRAINING
# ============================================================================
print("\n[8/9] Training...")
print("="*80)

best_val_f1 = 0.0
history = {'train_loss': [], 'train_acc': [], 'val_acc': [], 'val_f1': []}
patience_counter = 0

total_start_time = time.time()  # ✅ ADDED: Total time tracking

for epoch in range(Config.EPOCHS):
    epoch_start_time = time.time()  # ✅ ADDED: Epoch time tracking
    
    print(f'\n{"="*70}')
    print(f'EPOCH {epoch+1}/{Config.EPOCHS}')
    print(f'{"="*70}')
    
    # Training
    model.train()
    total_loss = 0.0
    train_preds, train_labels_list = [], []
    
    # ✅ IMPROVED: Better progress bar with batch counter
    pbar = tqdm(train_loader, desc='Training', total=len(train_loader))
    for batch_idx, batch in enumerate(pbar, 1):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels_batch = batch['labels'].to(device)
        
        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        preds = logits.argmax(dim=1).cpu().numpy()
        train_preds.extend(preds.tolist())
        train_labels_list.extend(labels_batch.cpu().numpy().tolist())
        
        # ✅ IMPROVED: Show average loss and batch progress
        avg_loss = total_loss / batch_idx
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'avg': f'{avg_loss:.4f}'})
    
    train_loss = total_loss / len(train_loader)
    train_acc = accuracy_score(train_labels_list, train_preds)
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    
    print(f'\n📊 Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f}')
    
    # Validation
    model.eval()
    val_preds, val_labels_list = [], []
    
    with torch.no_grad():
        for batch in tqdm(val_loader, desc='Validation'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels_batch = batch['labels'].to(device)
            
            logits = model(input_ids, attention_mask)
            val_preds.extend(logits.argmax(dim=1).cpu().numpy().tolist())
            val_labels_list.extend(labels_batch.cpu().numpy().tolist())
    
    val_acc = accuracy_score(val_labels_list, val_preds)
    val_f1 = f1_score(val_labels_list, val_preds, average='weighted')
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)
    
    # ✅ ADDED: Epoch timing
    epoch_time = time.time() - epoch_start_time
    
    # Calculate overfitting gap
    overfit_gap = train_acc - val_acc
    print(f'📊 Val Acc: {val_acc:.4f} | F1: {val_f1:.4f}')
    print(f'📉 Overfitting Gap: {overfit_gap:.4f} (Train-Val)', end='')
    
    if overfit_gap > 0.10:
        print("⚠️ Overfitting (large train > val gap)")
    elif overfit_gap > 0.05:
        print("⚡ Mild overfitting")
    elif overfit_gap < -0.05:
        print("🔵 Underfitting / early training phase")
    else:
        print("✅ Balanced generalization")
    
    # ✅ ADDED: Time info
    print(f'⏱️  Epoch time: {epoch_time/60:.1f} minutes')
    if epoch > 0:
        avg_epoch_time = (time.time() - total_start_time) / (epoch + 1)
        eta = avg_epoch_time * (Config.EPOCHS - epoch - 1)
        print(f'⏳ ETA: {eta/60:.1f} minutes ({eta/3600:.1f} hours)')
    
    # Save best model & early stopping
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        patience_counter = 0
        torch.save({
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'epoch': epoch,
            'val_f1': val_f1,
        }, 'best_bert_only_model.pt')
        print(f'✅ Saved best model (F1: {best_val_f1:.4f})')
    else:
        patience_counter += 1
        print(f'⏳ No improvement for {patience_counter} epoch(s)')
        
        if patience_counter >= Config.EARLY_STOPPING_PATIENCE:
            print(f'\n🛑 Early stopping triggered! Best Val F1: {best_val_f1:.4f}')
            break


[8/9] Training...

EPOCH 1/20


Training: 100%|███████████████████████████████████████████| 1536/1536 [05:32<00:00,  4.62it/s, loss=1.8573, avg=2.1737]



📊 Train Loss: 2.1737 | Acc: 0.2653


Validation: 100%|████████████████████████████████████████████████████████████████████| 768/768 [01:35<00:00,  8.00it/s]


📊 Val Acc: 0.7756 | F1: 0.7441
📉 Overfitting Gap: -0.5103 (Train-Val)🔵 Underfitting / early training phase
⏱️  Epoch time: 7.1 minutes
✅ Saved best model (F1: 0.7441)

EPOCH 2/20


...

📊 Val Acc: 0.8769 | F1: 0.8765
📉 Overfitting Gap: 0.0801 (Train-Val)⚡ Mild overfitting
⏱️  Epoch time: 7.1 minutes
⏳ ETA: 28.6 minutes (0.5 hours)
⏳ No improvement for 4 epoch(s)

EPOCH 17/20


Training: 100%|███████████████████████████████████████████| 1536/1536 [05:31<00:00,  4.63it/s, loss=0.5215, avg=0.6502]



📊 Train Loss: 0.6502 | Acc: 0.9580


Validation: 100%|████████████████████████████████████████████████████████████████████| 768/768 [01:34<00:00,  8.08it/s]

📊 Val Acc: 0.8790 | F1: 0.8789
📉 Overfitting Gap: 0.0790 (Train-Val)⚡ Mild overfitting
⏱️  Epoch time: 7.1 minutes
⏳ ETA: 21.5 minutes (0.4 hours)
⏳ No improvement for 5 epoch(s)

🛑 Early stopping triggered! Best Val F1: 0.8794


In [9]:
# ============================================================================
# SECTION 9: TESTING
# ============================================================================
print("\n" + "="*80)
print("TESTING ON UNLABELED DATA (SIMULATING REAL SEMI-SUPERVISED SCENARIO)")
print("="*80)

checkpoint = torch.load('best_bert_only_model.pt', weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"✓ Loaded best model from epoch {checkpoint['epoch']+1}")

model.eval()
test_preds, test_labels_list = [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc='Testing'):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels_batch = batch['labels'].to(device)
        
        logits = model(input_ids, attention_mask)
        test_preds.extend(logits.argmax(dim=1).cpu().numpy().tolist())
        test_labels_list.extend(labels_batch.cpu().numpy().tolist())

test_acc = accuracy_score(test_labels_list, test_preds)
test_f1 = f1_score(test_labels_list, test_preds, average='weighted')

print(f"\n{'='*70}")
print(f"BERT-ONLY FINAL TEST RESULTS (Trained on 10% labeled)")
print(f"{'='*70}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test F1 Score: {test_f1:.4f}")
print(f"{'='*70}")

print("\nClassification Report:")
print(classification_report(test_labels_list, test_preds, target_names=le.classes_, digits=4))

total_time = time.time() - total_start_time
print(f"\n{'='*70}")
print("✅ Training Complete!")
print(f"✅ Total training time: {total_time/60:.1f} minutes ({total_time/3600:.1f} hours)")
print(f"✅ Masks saved: train_mask.npy, val_mask.npy, test_mask.npy")
print(f"✅ Model saved: best_bert_only_model.pt")
print(f"{'='*70}")


TESTING ON UNLABELED DATA (SIMULATING REAL SEMI-SUPERVISED SCENARIO)
✓ Loaded best model from epoch 12


Testing: 100%|███████████████████████████████████████████████████████████████████| 13050/13050 [26:59<00:00,  8.06it/s]



BERT-ONLY FINAL TEST RESULTS (Trained on 10% labeled)
Test Accuracy: 0.8777
Test F1 Score: 0.8774

Classification Report:
                        precision    recall  f1-score   support

                   art     0.7936    0.7312    0.7611      8349
                 crime     0.9386    0.9381    0.9383     12742
               economy     0.8756    0.8731    0.8743     12244
             education     0.9007    0.9021    0.9014      7540
         entertainment     0.7949    0.8380    0.8159     10024
                global     0.8794    0.8727    0.8761     12296
                health     0.8501    0.8848    0.8671     12353
              politics     0.9175    0.8711    0.8937     12561
science and technology     0.8283    0.8416    0.8349      6734
                sports     0.9584    0.9845    0.9713      9552

              accuracy                         0.8777    104395
             macro avg     0.8737    0.8737    0.8734    104395
          weighted avg     0.8778    0.8777